# CS460 Anti-UAV Detection Pipeline — Cloud GPU Launcher (Colab / Kaggle)

**Title:** *Small Targets at Speed: Input Resolution and RPN Anchor Scale Sensitivity in Vision-Based Anti-UAV Detection*  
**Repository:** `https://github.com/Lc-wing/CV_final_project.git`

This thin launcher notebook auto-detects **Google Colab** (mounting Google Drive at `/content/drive/MyDrive/CV_project` so datasets, checkpoints, and raw CSV logs survive disconnects) or **Kaggle Notebooks** (`/kaggle/working/CV_project`), then invokes the exact CLI commands documented in `README.md` for all three controlled runs:

1. `res_800` *(Baseline)*: `min_size=800, max_size=1333`, anchors `(32, 64, 128, 256, 512)`
2. `res_480` *(Variant 1)*: `min_size=480, max_size=800`, anchors `(32, 64, 128, 256, 512)`
3. `res_480_small_anchor` *(Variant 2)*: `min_size=480, max_size=800`, anchors `(16, 32, 64, 128, 256)`

## 1. Environment Auto-Detection & Persistent Storage Setup

In [ ]:
import os
import subprocess
from pathlib import Path

IN_COLAB = False
IN_KAGGLE = Path("/kaggle/working").exists()

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    STORAGE_ROOT = Path("/content/drive/MyDrive/CV_project")
    REPO_DIR = Path("/content/CV_final_project")
    PLATFORM = "Google Colab"
elif IN_KAGGLE:
    STORAGE_ROOT = Path("/kaggle/working/CV_project")
    REPO_DIR = Path("/kaggle/working/CV_final_project")
    PLATFORM = "Kaggle"
else:
    STORAGE_ROOT = Path.cwd()
    REPO_DIR = Path.cwd()
    PLATFORM = "Local"

DATA_DIR = STORAGE_ROOT / "data" / "dut_anti_uav"
CKPT_DIR = STORAGE_ROOT / "checkpoints"
RESULTS_DIR = STORAGE_ROOT / "results"
LOG_DIR = RESULTS_DIR / "logs"
METRICS_DIR = RESULTS_DIR / "metrics"
FIGURES_DIR = RESULTS_DIR / "figures"

for d in (DATA_DIR, CKPT_DIR, LOG_DIR, METRICS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

os.environ["STORAGE_ROOT"] = str(STORAGE_ROOT)
os.environ["DATA_DIR"] = str(DATA_DIR)
os.environ["CKPT_DIR"] = str(CKPT_DIR)
os.environ["RESULTS_DIR"] = str(RESULTS_DIR)
os.environ["LOG_DIR"] = str(LOG_DIR)
os.environ["METRICS_DIR"] = str(METRICS_DIR)
os.environ["FIGURES_DIR"] = str(FIGURES_DIR)

print(f"Platform:     {PLATFORM}")
print(f"REPO_DIR:     {REPO_DIR}")
print(f"STORAGE_ROOT: {STORAGE_ROOT}")
print(f"DATA_DIR:     {DATA_DIR}")
print(f"CKPT_DIR:     {CKPT_DIR}")
print(f"RESULTS_DIR:  {RESULTS_DIR}")

subprocess.run(["nvidia-smi"], check=False)

## 2. Clone / Pull Repository & Install Dependencies

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/Lc-wing/CV_final_project.git"
BRANCH = "main"

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import torch
import torchvision

print(f"Working dir:  {os.getcwd()}")
print(f"PyTorch:      {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device:   {torch.cuda.get_device_name(0)}")
print(f"TorchVision:  {torchvision.__version__}")

## 3. Download DUT Anti-UAV Dataset & Run `train + val` EDA

Downloads `aegean-ai/dut-anti-uav` (Apache-2.0) into `$DATA_DIR` and computes original-coordinate scale bin edges strictly from `train + val` (never touching `test`), emitting `eda_train_val.json` and sibling `scale_bins.json`.

In [ ]:
import json
import subprocess
import sys

eda_out = METRICS_DIR / "eda_train_val.json"
cmd = [
    sys.executable, "-m", "src.prepare_data",
    "--data-dir", str(DATA_DIR),
    "--eda-out", str(eda_out),
]
# Pass --skip-download automatically if train/val/test img directories are already populated
if all((DATA_DIR / split / "img").is_dir() and any((DATA_DIR / split / "img").iterdir()) for split in ("train", "val", "test")):
    cmd.append("--skip-download")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

scale_bins_path = METRICS_DIR / "scale_bins.json"
print("scale_bins.json:")
print(json.dumps(json.loads(scale_bins_path.read_text(encoding="utf-8")), indent=2))

## 4. Fast Smoke-Test & Timing Verification (Run First!)

Verifies CUDA AMP forward/backward/step execution, CSV logging, and checkpoint saving in under a minute before launching the 12-epoch training runs.

In [ ]:
import subprocess
import sys
import time

smoke_logs = RESULTS_DIR / "smoke_logs"
smoke_ckpts = STORAGE_ROOT / "smoke_checkpoints"

t0 = time.perf_counter()
subprocess.run([
    sys.executable, "-m", "src.train",
    "--config", "configs/res_800.yaml",
    "--data-dir", str(DATA_DIR),
    "--log-dir", str(smoke_logs),
    "--checkpoint-dir", str(smoke_ckpts),
    "--smoke-test",
], check=True)
print(f"Smoke test completed cleanly in {time.perf_counter() - t0:.1f}s")

## 5. Train All 3 Controlled Runs (Automatic `--resume` on Disconnect)

Each run writes raw step/epoch CSV logs to `$LOG_DIR` and saves `<run>_last.pt` and `<run>_best.pt` (selected strictly by validation `AP50`) to `$CKPT_DIR` after every epoch. If Colab disconnects mid-run, re-running this cell automatically passes `--resume` whenever `<run>_last.pt` already exists.

In [ ]:
import subprocess
import sys

RUN_NAMES = ["res_800", "res_480", "res_480_small_anchor"]

for run_name in RUN_NAMES:
    cmd = [
        sys.executable, "-m", "src.train",
        "--config", f"configs/{run_name}.yaml",
        "--data-dir", str(DATA_DIR),
        "--log-dir", str(LOG_DIR),
        "--checkpoint-dir", str(CKPT_DIR),
    ]
    if (CKPT_DIR / f"{run_name}_last.pt").is_file():
        cmd.append("--resume")
        print(f"[{run_name}] Found existing _last.pt -> resuming")
    else:
        print(f"[{run_name}] Starting fresh training run")

    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

## 6. Test-Split Scale-Stratified Evaluation & Latency Benchmark

In [ ]:
import subprocess
import sys

scale_bins_path = METRICS_DIR / "scale_bins.json"

for run_name in ["res_800", "res_480", "res_480_small_anchor"]:
    cmd = [
        sys.executable, "-m", "src.eval",
        "--config", f"configs/{run_name}.yaml",
        "--checkpoint", str(CKPT_DIR / f"{run_name}_best.pt"),
        "--data-dir", str(DATA_DIR),
        "--split", "test",
        "--scale-bins", str(scale_bins_path),
        "--out", str(METRICS_DIR / f"{run_name}_test.json"),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

## 7. Regenerate Training Loss Curve & Scale-Recall / Latency Comparison Figures

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, "-m", "src.plot_curves",
    "--logs",
    str(LOG_DIR / "res_800_epoch_log.csv"),
    str(LOG_DIR / "res_480_epoch_log.csv"),
    str(LOG_DIR / "res_480_small_anchor_epoch_log.csv"),
    "--loss-out", str(FIGURES_DIR / "training_loss_curves.png"),
    "--x-axis", "epoch",
    "--eval-jsons",
    str(METRICS_DIR / "res_800_test.json"),
    str(METRICS_DIR / "res_480_test.json"),
    str(METRICS_DIR / "res_480_small_anchor_test.json"),
    "--compare-out", str(FIGURES_DIR / "scale_recall_and_latency.png"),
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

## 8. Small Test-Pack Predictions, Comparison Visuals & Single-Image Spot Check

In [ ]:
import subprocess
import sys
from pathlib import Path

test_pack_dir = Path("submission_pack/test_pack")
out_csv = Path("submission_pack/predictions/test_pack_predictions.csv")
visuals_dir = Path("submission_pack/predictions/visuals")

if (test_pack_dir / "img").is_dir() and any((test_pack_dir / "img").iterdir()):
    pack_cmd = [
        sys.executable, "-m", "src.predict_pack",
        "--run", "configs/res_800.yaml", str(CKPT_DIR / "res_800_best.pt"),
        "--run", "configs/res_480.yaml", str(CKPT_DIR / "res_480_best.pt"),
        "--run", "configs/res_480_small_anchor.yaml", str(CKPT_DIR / "res_480_small_anchor_best.pt"),
        "--test-pack-dir", str(test_pack_dir),
        "--out-csv", str(out_csv),
        "--visuals-dir", str(visuals_dir),
    ]
    print("Running:", " ".join(pack_cmd))
    subprocess.run(pack_cmd, check=True)

    sample_img = sorted((test_pack_dir / "img").glob("*"))[0]
    single_cmd = [
        sys.executable, "-m", "src.predict_single",
        "--config", "configs/res_800.yaml",
        "--checkpoint", str(CKPT_DIR / "res_800_best.pt"),
        "--image", str(sample_img),
        "--score-thresh", "0.50",
        "--out-image", str(FIGURES_DIR / f"spot_check_{sample_img.stem}.png"),
    ]
    print("Running:", " ".join(single_cmd))
    subprocess.run(single_cmd, check=True)
else:
    print("Populate submission_pack/test_pack/{img,xml} with curated test samples first.")

## 9. Print Checkpoint SHA-256 Checksums (for `checkpoints_manifest.json`)

In [ ]:
from scripts.download_checkpoints import compute_sha256

for run_name in ["res_800", "res_480", "res_480_small_anchor"]:
    ckpt = CKPT_DIR / f"{run_name}_best.pt"
    if ckpt.is_file():
        print(f"{ckpt.name}: {compute_sha256(ckpt)} ({ckpt.stat().st_size / (1024**2):.2f} MB)")
    else:
        print(f"{ckpt.name}: NOT FOUND")